# Model RANDOM FOREST

In [ ]:
# IMPORT, CONFIG, AND LOAD DATASET
# ============================================================
import os
import time
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.stats import rankdata

RANDOM_STATE = 42
N_FOLDS = 5
TOP_N = 10          
HIGH_CORR_THRESHOLD = 0.98

DATA_DIR = "/Users/annya/DAC/Dataset/Processed"
OUT_DIR = "/Users/annya/DAC/outputs/submissions"

os.makedirs(OUT_DIR, exist_ok=True)

X_train = pd.read_csv(f"{DATA_DIR}/X_train.csv")
X_val = pd.read_csv(f"{DATA_DIR}/X_val.csv")
X_test = pd.read_csv(f"{DATA_DIR}/X_test.csv")

y_train = pd.read_csv(f"{DATA_DIR}/y_train.csv").squeeze().values
y_val = pd.read_csv(f"{DATA_DIR}/y_val.csv").squeeze().values

test_ids = pd.read_csv(f"{DATA_DIR}/test_ids.csv").squeeze()

X_full = pd.concat([X_train, X_val], axis=0).reset_index(drop=True)
y_full = np.concatenate([y_train, y_val])

print("Full CV pool:", X_full.shape)
print("Test        :", X_test.shape)

# RF SEARCH SPACE
# ============================================================
RF_CONFIGS_A = [

    {"n_estimators": 1000, "max_depth": 40, "max_features": 0.5,
     "min_samples_leaf": 1, "min_samples_split": 2, "max_samples": 1.0,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 50, "max_features": 0.5,
     "min_samples_leaf": 1, "min_samples_split": 2, "max_samples": 0.9,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 35, "max_features": 0.6,
     "min_samples_leaf": 1, "min_samples_split": 2, "max_samples": 0.8,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 30, "max_features": 0.5,
     "min_samples_leaf": 2, "min_samples_split": 4, "max_samples": 0.8,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4, "max_samples": 0.9,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 25, "max_features": 0.8,
     "min_samples_leaf": 2, "min_samples_split": 4, "max_samples": 1.0,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 30, "max_features": 0.3,
     "min_samples_leaf": 1, "min_samples_split": 2, "max_samples": 0.8,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 35, "max_features": 0.4,
     "min_samples_leaf": 2, "min_samples_split": 3, "max_samples": 0.7,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 40, "max_features": 0.4,
     "min_samples_leaf": 2, "min_samples_split": 4, "max_samples": 0.8,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 20, "max_features": 0.8,
     "min_samples_leaf": 4, "min_samples_split": 8, "max_samples": 0.9,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 25, "max_features": 0.7,
     "min_samples_leaf": 5, "min_samples_split": 10, "max_samples": 0.8,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4, "max_samples": 0.9,
     "criterion": "gini", "class_weight": None},

    {"n_estimators": 1000, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4, "max_samples": 0.9,
     "criterion": "gini", "class_weight": "balanced"},

    {"n_estimators": 1000, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4, "max_samples": 0.9,
     "criterion": "gini", "class_weight": {0: 1, 1: 1.5}},

    {"n_estimators": 1000, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4, "max_samples": 0.9,
     "criterion": "gini", "class_weight": {0: 1, 1: 2}},

    {"n_estimators": 1000, "max_depth": 35, "max_features": 0.6,
     "min_samples_leaf": 1, "min_samples_split": 2, "max_samples": 0.8,
     "criterion": "entropy", "class_weight": "balanced_subsample"},

    {"n_estimators": 1000, "max_depth": 35, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4, "max_samples": 0.8,
     "criterion": "log_loss", "class_weight": "balanced_subsample"},
]

RF_CONFIGS_B = [

    {"n_estimators": 700, "max_depth": 23, "max_features": 0.8,
     "min_samples_leaf": 3, "min_samples_split": 6,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 27, "max_features": 0.8,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.8,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 35, "max_features": 0.8,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 25, "max_features": 0.5,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.5,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 35, "max_features": 0.5,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 25, "max_features": 0.6,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 35, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.8,
     "min_samples_leaf": 1, "min_samples_split": 2,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 35, "max_features": 0.7,
     "min_samples_leaf": 1, "min_samples_split": 2,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 25, "max_features": 0.8,
     "min_samples_leaf": 4, "min_samples_split": 8,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 5, "min_samples_split": 10,
     "criterion": "gini", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.8,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": "balanced"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.8,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "gini", "class_weight": None},

    {"n_estimators": 700, "max_depth": 25, "max_features": 0.8,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "entropy", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "entropy", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 25, "max_features": 0.8,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "log_loss", "class_weight": "balanced_subsample"},

    {"n_estimators": 700, "max_depth": 30, "max_features": 0.7,
     "min_samples_leaf": 2, "min_samples_split": 4,
     "criterion": "log_loss", "class_weight": "balanced_subsample"},
]

RF_CONFIGS = RF_CONFIGS_A + RF_CONFIGS_B

print(f"\nTotal RF configs digabung: {len(RF_CONFIGS)} "
      f"({len(RF_CONFIGS_A)} dari set A + {len(RF_CONFIGS_B)} dari set B)")

# K-FOLD CV TRAINING (OOF)
# ============================================================
skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

results = []
oof_preds_all = []
test_preds_all = []

overall_start = time.time()

for i, config in enumerate(RF_CONFIGS):

    print("\n" + "=" * 80)
    print(f"RF {i + 1}/{len(RF_CONFIGS)}")
    print("=" * 80)
    print(config)

    start = time.time()

    oof_pred = np.zeros(len(X_full))
    test_pred_folds = np.zeros((N_FOLDS, len(X_test)))
    fold_aucs = []

    for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X_full, y_full)):

        seed = RANDOM_STATE + i * 100 + fold_idx

        model = RandomForestClassifier(
            random_state=seed,
            n_jobs=-1,
            bootstrap=True,
            **config
        )

        model.fit(X_full.iloc[tr_idx], y_full[tr_idx])

        val_prob = model.predict_proba(X_full.iloc[va_idx])[:, 1]
        oof_pred[va_idx] = val_prob

        fold_auc = roc_auc_score(y_full[va_idx], val_prob)
        fold_aucs.append(fold_auc)

        test_pred_folds[fold_idx] = model.predict_proba(X_test)[:, 1]

        print(f"  fold {fold_idx + 1}/{N_FOLDS} AUC = {fold_auc:.7f}")

    elapsed = time.time() - start

    oof_auc = roc_auc_score(y_full, oof_pred)
    fold_std = np.std(fold_aucs)

    print(
        f"OOF AUC = {oof_auc:.7f} | "
        f"fold mean = {np.mean(fold_aucs):.7f} | "
        f"fold std = {fold_std:.7f} | "
        f"time = {elapsed:.1f}s"
    )

    oof_preds_all.append(oof_pred)
    test_preds_all.append(test_pred_folds.mean(axis=0))

    results.append({
        "id": i,
        "oof_auc": oof_auc,
        "fold_mean_auc": np.mean(fold_aucs),
        "fold_std_auc": fold_std,
        **config
    })

print(f"\nTotal training time: {(time.time() - overall_start) / 60:.1f} min")

# RESULTS TABLE
# ============================================================
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("oof_auc", ascending=False).reset_index(drop=True)

print("\n" + "=" * 80)
print("RESULTS (sorted by OOF AUC)")
print("=" * 80)
print(
    results_df[
        [
            "id", "oof_auc", "fold_mean_auc", "fold_std_auc",
            "n_estimators", "max_depth", "max_features",
            "min_samples_leaf", "min_samples_split", "criterion",
            "class_weight"
        ]
    ].head(15).to_string(index=False)
)

# TOP MODELS (by OOF AUC)
# ============================================================
top_ids = results_df.head(TOP_N)["id"].astype(int).tolist()

top_oof = [oof_preds_all[i] for i in top_ids]
top_test = [test_preds_all[i] for i in top_ids]
top_aucs = np.array([
    results_df.loc[results_df["id"] == i, "oof_auc"].values[0]
    for i in top_ids
])

# CORRELATION CHECK ANTAR TOP MODELS
# ============================================================
oof_matrix = np.column_stack(top_oof)
corr_matrix = np.corrcoef(oof_matrix.T)

print("\n" + "=" * 80)
print(f"CORRELATION MATRIX (top {TOP_N} models, by OOF prediction)")
print("=" * 80)

corr_df = pd.DataFrame(
    corr_matrix,
    index=[f"m{i}" for i in top_ids],
    columns=[f"m{i}" for i in top_ids]
)
print(corr_df.round(4).to_string())

high_corr_pairs = []
for a in range(len(top_ids)):
    for b in range(a + 1, len(top_ids)):
        if corr_matrix[a, b] > HIGH_CORR_THRESHOLD:
            high_corr_pairs.append((top_ids[a], top_ids[b], corr_matrix[a, b]))

if high_corr_pairs:
    print(f"\nPasangan model dengan korelasi > {HIGH_CORR_THRESHOLD} (kandidat dibuang):")
    for a, b, c in high_corr_pairs:
        print(f"  model {a} <-> model {b} : corr = {c:.4f}")
else:
    print(f"\nTidak ada pasangan model dengan korelasi > {HIGH_CORR_THRESHOLD}.")

# ENSEMBLE METHODS (OOF)
# ============================================================
# A. Simple probability average 
simple_oof = np.mean(oof_matrix, axis=1)
simple_auc = roc_auc_score(y_full, simple_oof)

# B. Rank ensemble 
rank_matrix = np.column_stack([rankdata(p) for p in top_oof])
rank_matrix = rank_matrix / len(y_full)
rank_oof = np.mean(rank_matrix, axis=1)
rank_auc = roc_auc_score(y_full, rank_oof)

# C. Weighted rank ensemble 
weights = np.sqrt(top_aucs - top_aucs.min() + 1e-6)
weights = weights / weights.sum()

weighted_rank_oof = np.average(rank_matrix, axis=1, weights=weights)
weighted_rank_auc = roc_auc_score(y_full, weighted_rank_oof)


# PER-FOLD STABILITY CHECK
# ============================================================
print("\n" + "=" * 80)
print("PER-FOLD ENSEMBLE COMPARISON (stability check)")
print("=" * 80)

fold_assignment = np.zeros(len(y_full), dtype=int)
for fold_idx, (_, va_idx) in enumerate(skf.split(X_full, y_full)):
    fold_assignment[va_idx] = fold_idx

per_fold_records = []

for fold_idx in range(N_FOLDS):
    mask = fold_assignment == fold_idx
    y_fold = y_full[mask]

    best_ind_fold = max(roc_auc_score(y_fold, p[mask]) for p in top_oof)
    simple_fold = roc_auc_score(y_fold, simple_oof[mask])
    rank_fold = roc_auc_score(y_fold, rank_oof[mask])
    wrank_fold = roc_auc_score(y_fold, weighted_rank_oof[mask])

    per_fold_records.append({
        "fold": fold_idx,
        "best_individual": best_ind_fold,
        "simple_avg": simple_fold,
        "rank_ens": rank_fold,
        "weighted_rank_ens": wrank_fold,
    })

per_fold_df = pd.DataFrame(per_fold_records)
print(per_fold_df.round(6).to_string(index=False))
print("\nMean +- std per method across folds:")
for col in ["best_individual", "simple_avg", "rank_ens", "weighted_rank_ens"]:
    print(f"  {col:20s}: {per_fold_df[col].mean():.6f} +- {per_fold_df[col].std():.6f}")

print(
    "\n>> Kalau 'weighted_rank_ens' TIDAK konsisten menang di "
    "mayoritas fold (misal cuma menang di 2/5), berarti gap-nya "
    "kemungkinan noise -- lebih aman pakai rank_ens atau simple_avg."
)

# OVERALL OOF SUMMARY
# ============================================================
print("\n" + "=" * 80)
print("OVERALL OOF ENSEMBLE RESULTS")
print("=" * 80)
print(f"Best individual OOF AUC : {top_aucs.max():.7f}")
print(f"Simple probability       : {simple_auc:.7f}")
print(f"Rank ensemble            : {rank_auc:.7f}")
print(f"Weighted rank            : {weighted_rank_auc:.7f}")


# TEST PREDICTIONS
# ============================================================
test_matrix = np.column_stack(top_test)

# --- Best individual---
best_individual_test = top_test[0]

# --- Simple average ---
test_simple = np.mean(test_matrix, axis=1)

# --- Rank ensemble ---
test_rank_matrix = np.column_stack([
    rankdata(test_matrix[:, j]) for j in range(test_matrix.shape[1])
])
test_rank_matrix = test_rank_matrix / len(X_test)
test_rank = np.mean(test_rank_matrix, axis=1)

# --- Weighted rank ---
test_weighted_rank = np.average(test_rank_matrix, axis=1, weights=weights)


# SAVE SUBMISSIONS FOR EVERY TEST PREDICTION
# ============================================================
def make_submission(pred):
    return pd.DataFrame({
        "claim_id": test_ids,
        "fraud_probability": np.clip(pred, 0, 1)
    })

best_path = f"{OUT_DIR}/rf_merged_best_individual.csv"
simple_path = f"{OUT_DIR}/rf_merged_probability_ensemble.csv"
rank_path = f"{OUT_DIR}/rf_merged_rank_ensemble.csv"
weighted_rank_path = f"{OUT_DIR}/rf_merged_weighted_rank_ensemble.csv"

make_submission(best_individual_test).to_csv(best_path, index=False)
make_submission(test_simple).to_csv(simple_path, index=False)
make_submission(test_rank).to_csv(rank_path, index=False)
make_submission(test_weighted_rank).to_csv(weighted_rank_path, index=False)

print("\n" + "=" * 80)
print("DONE")
print("=" * 80)
print(f"Best individual config id : {top_ids[0]}  (OOF AUC = {top_aucs[0]:.7f})")
print(best_path)
print(simple_path)
print(rank_path)
print(weighted_rank_path)

Full CV pool: (160174, 53)
Test        : (40043, 53)

Total RF configs digabung: 38 (17 dari set A + 21 dari set B)

RF 1/38
{'n_estimators': 1000, 'max_depth': 40, 'max_features': 0.5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'max_samples': 1.0, 'criterion': 'gini', 'class_weight': 'balanced_subsample'}
  fold 1/5 AUC = 0.7977972
  fold 2/5 AUC = 0.7948334
  fold 3/5 AUC = 0.7941978
  fold 4/5 AUC = 0.7972847
  fold 5/5 AUC = 0.7960702
OOF AUC = 0.7960312 | fold mean = 0.7960367 | fold std = 0.0013775 | time = 206.8s

RF 2/38
{'n_estimators': 1000, 'max_depth': 50, 'max_features': 0.5, 'min_samples_leaf': 1, 'min_samples_split': 2, 'max_samples': 0.9, 'criterion': 'gini', 'class_weight': 'balanced_subsample'}
  fold 1/5 AUC = 0.7982393
  fold 2/5 AUC = 0.7949452
  fold 3/5 AUC = 0.7947445
  fold 4/5 AUC = 0.7979915
  fold 5/5 AUC = 0.7967549
OOF AUC = 0.7965321 | fold mean = 0.7965351 | fold std = 0.0014702 | time = 202.5s

RF 3/38
{'n_estimators': 1000, 'max_depth': 35, 'max_fe

Menggunakan /Users/annya/DAC/outputs/submissions/rf_merged_weighted_rank_ensemble.csv dikarenakan mdoel tersebut adalah model dengan hasil terbaik ke submission

In [14]:
np.save("/Users/annya/DAC/results/predictions/y_full.npy", y_full)
np.save("/Users/annya/DAC/results/predictions/weighted_rank_oof.npy", weighted_rank_oof)
np.save("/Users/annya/DAC/results/predictions/test_weighted_rank.npy", test_weighted_rank)

print("Variabel berhasil disimpan untuk dipakai di notebook Task C & D")

Variabel berhasil disimpan untuk dipakai di notebook Task C & D
